In [ ]:
import os
import subprocess

REPO_DIR = "/content/kybelix-gid-segmentation"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/ysfztpp/kybelix-gid-segmentation.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
print("Repo ready:", os.getcwd())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

def _has_data(root: Path) -> bool:
    return (root / "unmasked").is_dir() and (root / "masked").is_dir()


def _resolve_data_root(preferred_root: Path, search_root: Path):
    if _has_data(preferred_root):
        return preferred_root
    if _has_data(search_root):
        return search_root
    if search_root.is_dir():
        for child in sorted(search_root.iterdir()):
            if child.is_dir() and _has_data(child):
                return child
    return None

# Toggle here:
# - drive_stream: old mode, reads directly from Drive folders.
# - zip_local_cache: copy zip from Drive to /content and train from local disk.
DATA_SOURCE_MODE = "zip_local_cache"
SHORTCUT_NAME = "phase1data"

os.environ["KYBELIX_DATA_SOURCE_MODE"] = DATA_SOURCE_MODE
os.environ["KYBELIX_SHORTCUT_NAME"] = SHORTCUT_NAME

if DATA_SOURCE_MODE == "zip_local_cache":
    drive_zip = Path(f"/content/drive/MyDrive/{SHORTCUT_NAME}.zip")
    local_zip = Path(f"/content/{SHORTCUT_NAME}.zip")
    local_root = Path(f"/content/{SHORTCUT_NAME}")

    if not drive_zip.exists():
        raise FileNotFoundError(f"Zip not found in Drive: {drive_zip}")

    needs_copy = (not local_zip.exists()) or (local_zip.stat().st_size != drive_zip.stat().st_size)
    force_extract = False
    if needs_copy:
        shutil.copy2(drive_zip, local_zip)
        force_extract = True
        print(f"Copied zip to: {local_zip}")
    else:
        print(f"Using existing local zip: {local_zip}")

    data_root = _resolve_data_root(local_root, Path("/content"))
    if force_extract or data_root is None:
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall("/content")
        print("Extracted zip into /content")
        data_root = _resolve_data_root(local_root, Path("/content"))

    if data_root is None:
        raise FileNotFoundError(
            f"Could not find unmasked/masked folders after extracting: {local_zip}"
        )

    os.environ["KYBELIX_DATA_ROOT"] = str(data_root)
    print(f"Using extracted dataset root: {data_root}")
else:
    os.environ["KYBELIX_DATA_ROOT"] = f"/content/drive/MyDrive/{SHORTCUT_NAME}"

data_root = Path(os.environ["KYBELIX_DATA_ROOT"])
if not ((data_root / "unmasked").is_dir() and (data_root / "masked").is_dir()):
    raise FileNotFoundError(f"Dataset folders not found under: {data_root}")

print("KYBELIX_DATA_SOURCE_MODE =", os.environ["KYBELIX_DATA_SOURCE_MODE"])
print("KYBELIX_DATA_ROOT =", os.environ["KYBELIX_DATA_ROOT"])


In [ ]:
%cd /content/kybelix-gid-segmentation
!python3 -c "from config import Config; print('BASE_PATH:', Config.BASE_PATH); print('DATA_SOURCE_MODE:', Config.DATA_SOURCE_MODE)"


In [ ]:
%cd /content/kybelix-gid-segmentation
!python3 train.py
